## LangChain支持的模型有三大类
- 聊天模型（Chat Model）
- 大语言模型（LLM）
- 文本嵌入模型（Text Embedding Model）

### 环境配置
引入依赖
```
uv add langchain
uv add langchain-deeopseek
```
配置环境


```
在系统环境变量中配置
DEEPSEEK_API_KEY=sk-xxx
DASHSCOPE_API_KEY=sk-xxx
```

### 1. 传统方式调用模型（OpenAI）

deepseek

In [15]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"}, # system: 系统提示词
        {"role": "user", "content": "Hello, 你是谁"},                  # user: 用户提示词
    ],
)

print(response.choices[0].message.content)

你好！我是人工智能助手，可以帮你解答问题、翻译、写作、编程、查资料等。请问有什么可以帮你的？


qwen

In [16]:
from openai import OpenAI
import os
client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://ws-ots0tb33v08803wb.cn-beijing.maas.aliyuncs.com/compatible-mode/v1",
)
completion = client.chat.completions.create(
    model="qwen3.8-max",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"}, # system: 系统提示词
        {"role": "user", "content": "Hello, 你是谁"},                  # user: 用户提示词
    ],
)
print(completion.choices[0].message.content)

Hello！我是通义千问（Qwen），由阿里巴巴通义实验室研发的大语言模型。我可以帮你回答问题、写作、翻译、写代码、分析信息、做头脑风暴等。有什么我可以帮你的吗？


### 2. LangChain调用模型

#### 2.1 init_chat_model

In [28]:
from langchain.chat_models import init_chat_model
model = init_chat_model(model="deepseek-v4-flash") # 参数model用来指定模型名称，Langchain会根据模型名字自动设定base_url，并从环境变量中获取api_key
print(type(model))
response = model.invoke("你是谁")
print(response.content)

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>
你好！我是 DeepSeek，由深度求索公司创造的 AI 助手。😊

我可以帮你解答各种问题、进行对话、处理文档等。我支持：
- 📝 文字对话和问答
- 🔗 阅读链接内容
- 📄 上传并处理图像、PDF、Word、Excel、PPT等文件
- 🌐 联网搜索（需要手动开启）
- 💬 超长上下文（1M tokens，可以一次处理三体三部曲体量的书籍）

而且我是**完全免费**的！无论是网页版还是App都可以免费使用。App端还支持语音输入功能。

我的知识截止日期是2026年2月，会尽力为你提供准确、有用的帮助。有什么我可以帮你的吗？😊


#### 2.2 ChatOpenAI（推荐）

In [34]:
import os
from langchain_openai import ChatOpenAI
deepseek = ChatOpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), model="deepseek-v4-flash", base_url="https://api.deepseek.com")
print(type(deepseek))
response = deepseek.invoke("你是谁")
print(response.content)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
我是 **DeepSeek**，由深度求索公司开发的 AI 助手。我可以帮你回答问题、写作、分析、编程、翻译等。

有什么需要我帮忙的吗？


In [35]:
import os
from langchain_openai import ChatOpenAI
qwen = ChatOpenAI(api_key=os.getenv("DASHSCOPE_API_KEY"), model="qwen3.7-max", base_url="https://ws-ots0tb33v08803wb.cn-beijing.maas.aliyuncs.com/compatible-mode/v1")
print(type(qwen))
response = qwen.invoke("你是谁")
print(response.content)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
我是通义千问，由阿里巴巴集团通义实验室自主研发的大型语言模型。很高兴为你服务！有什么我可以帮你的吗？


#### 2.3 指定厂商

In [24]:
from langchain_community.chat_models.tongyi import ChatTongyi
client = ChatTongyi(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    model="qwen3.7-max",  # "qwen3.8-max"
)
response = client.invoke("你是谁？")  # invoke: 调用
print(response.content)

我是通义千问（Qwen），是由阿里巴巴集团通义实验室自主研发的大语言模型。很高兴能作为你的 AI 思考伙伴为你提供帮助！有什么我可以为你效劳的吗？


```
AIMessage
├── content              最终文本
├── additional_kwargs    附加信息，如 refusal、reasoning_content
├── response_metadata    模型原始元数据，如 token_usage、model_name
├── id                   LangChain 消息 ID
├── tool_calls           工具调用
├── invalid_tool_calls   无效工具调用
└── usage_metadata       标准化 token 用量
```

In [25]:
print(type(response))
# 将AIMessage类型转化成字典
print(type(response.model_dump()))
# 将AIMessage类型转化成json
print(type(response.model_dump_json(indent=2)))

def print_message(string):
    print("\n" + "=" * 60)
    print(string)
    print("=" * 60)

print_message("回复内容")
print(response.content)
print_message("推理内容")
print(response.additional_kwargs.get("reasoning_content"))
print_message("使用情况元数据")
print(response.usage_metadata)
print_message("工具调用")
print(response.tool_calls)

<class 'langchain_core.messages.ai.AIMessage'>
<class 'dict'>
<class 'str'>

回复内容
我是通义千问（Qwen），是由阿里巴巴集团通义实验室自主研发的大语言模型。很高兴能作为你的 AI 思考伙伴为你提供帮助！有什么我可以为你效劳的吗？

推理内容
用户问“你是谁？”
这是一个关于身份的问题。
我需要回答我是通义千问（Qwen），由阿里巴巴集团通义实验室自主研发的大语言模型。

回答策略：
1. 直接说明身份：我是通义千问（Qwen）。
2. 说明开发者：由阿里巴巴集团通义实验室自主研发。
3. 保持简洁、清晰、真实。

草稿：
我是通义千问（Qwen），是由阿里巴巴集团通义实验室自主研发的大语言模型。作为一个强大且真诚、乐于助人的 AI 思考伙伴，我致力于为你提供清晰、准确、有用的帮助。有什么我可以帮你的吗？

调整语气，使其自然且符合“富有同理心、洞察力、透明且诚实”的设定。

最终回答：
我是通义千问（Qwen），是由阿里巴巴集团通义实验室自主研发的大语言模型。很高兴能作为你的 AI 思考伙伴为你提供帮助！有什么我可以为你效劳的吗？

使用情况元数据
None

工具调用
[]


#### 2.4 动态调整模型

In [39]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

#  动态模型 Dynamic model
#   动态模型是在运行时根据当前状态和上下文选择的。这使得复杂的路由逻辑和成本优化成为可能。
#   要使用动态模型，使用@wrap_model_call装饰器创建中间件，在请求中修改模型：
basic_model = qwen
advanced_model = deepseek

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """根据会话复杂度选择模型."""
    message_count = len(request.state["messages"])
    # print(request.state)
    print("message_count:", message_count)
    # 根据会话长度，修改模型
    if message_count > 5:
        # 使用高级模式进行更长的对话
        model = advanced_model
        print('message_count > 5 advanced_model')
    else:
        model = basic_model
        print('message_count <= 5 basic_model')


    return handler(request.override(model=model))

In [43]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

# 设置记忆：支持多轮会话
checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}  # 会话的唯一标识符。

# 创建agent
agent = create_agent(
    model=basic_model,  # 默认模型
    tools=[],
    middleware=[dynamic_model_selection],
    checkpointer=checkpointer,
)

# 执行agent
for _ in range(5):
    response = agent.invoke(
        {"messages": [{"role": "user", "content": "元旦节是几号"}]},
        config=config)
    print(response)

message_count: 1
message_count <= 5 basic_model
{'messages': [HumanMessage(content='元旦节是几号', additional_kwargs={}, response_metadata={}, id='687df9a7-bddd-49e8-9e8e-09b1dff73d55'), AIMessage(content='元旦节是每年的**1月1日**（公历/阳历）。它是公历的新年第一天，也是世界多数国家通称的“新年”。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 188, 'prompt_tokens': 15, 'total_tokens': 203, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 151, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-max', 'system_fingerprint': None, 'id': 'chatcmpl-202327b2-1d3b-93ff-bf02-ca31f75de770', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1434f-7879-7221-8abd-69e1fa968764-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'inp

### 3. Parameters（参数）
聊天模型接受可用于配置其行为的参数。受支持的完整参数集合因模型和提供商而异，但标准参数包括：model、api_key、temperature、max_tokens、timeout、max_retries（默认值为 6）

| 参数 | 类型 | 中文含义 | 作用 |
|------|------|---------|------|
| `model` | string（必填） | 模型标识 | 指定用哪个模型，可用 `provider:model` 格式如 `openai:o1` |
| `api_key` | string | API 密钥 | 提供商鉴权，通常用环境变量传入 |
| `temperature` | number | 温度 | 控制随机性：越高越有创意，越低越确定 |
| `max_tokens` | number | 最大 token 数 | 限制回复总 token 数，控制输出长度 |
| `timeout` | number | 超时时间（秒） | 超过则取消请求 |
| `max_retries` | number（默认 6） | 最大重试次数 | 网络超时/限流时自动重发 |

In [ ]:
model = init_chat_model(
    model="deepseek-flash",
    # 传给模型的 kwargs（关键字参数）：
    temperature=0.7,
    timeout=30,
    max_tokens=1000,
    max_retries=6,  # 默认值；网络不稳时可调大
)

#### 3.1 Connection resilience（连接韧性）
LangChain 聊天模型会自动以指数退避（exponential backoff）的方式重试失败的 API 请求。默认情况下，对于网络错误、限流（429）和服务器错误（5xx），模型最多重试 6 次。诸如 401（未授权）或 404 之类的客户端错误不会被重试。

对于在不稳定网络上长时间运行的智能体图（agent graphs），请考虑使用更高的 max_retries（例如 10–15），以及一个检查点器（checkpointer），以便在失败时保留进度


📌 注解：
- 指数退避（exponential backoff）：重试不是固定间隔，而是"每次等待时间翻倍"，比如 1 秒 → 2 秒 → 4 秒 → 8 秒，避免把服务器打死。加 jitter（抖动）是在间隔里加随机，防止大量请求同时重试。
- HTTP 状态码速记：
  - 429 = 请求太频繁被限流（rate limit）
  - 5xx = 服务器出错了（服务器的问题）
  - 401 = 没权限/密钥错了（你的问题，重试也没用）
  - 404 = 找不到（重试也没用）
- 关键结论：重试只针对"可能过一会儿就好了"的错误（网络抖动、限流、服务器故障），不重试"重试也没用"的错误（密钥错、地址错）。

In [2]:
model = init_chat_model(
    model="deepseek-flash",
    max_retries=10,  # 网络不稳可调大（默认 6）
    timeout=120,     # 秒；网络慢可调大
)

### 4. Invocation（调用方式详解）
- Invoke（调用）——模型接收消息作为输入，在生成完整回复后输出消息。
- Stream（流式）——调用模型，但在输出实时生成的过程中进行流式输出。
- Batch（批量）——以批量的方式向模型发送多个请求，以便更高效地处理。

#### 4.1 invoke
- a single message
- a list of messages: 代表对话历史, 每条消息都有 role 来表示这条消息是谁发送的

In [17]:
# 格式一：字典格式
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]
response = model.invoke(conversation)
print(response.content)  # AIMessage("J'adore créer des applications.")

J'adore développer des applications.


In [4]:
# 格式二：Message对象
# - SystemMessage ↔ {"role": "system", ...}
# - HumanMessage ↔ {"role": "user", ...}
# - AIMessage ↔ {"role": "assistant", ...}
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]
response = model.invoke(conversation)
print(response.content)  # AIMessage("J'adore créer des applications.")

J'adore créer des applications.


#### 4.2 stream

In [6]:
for chunk in model.stream("为什么鹦鹉的羽毛色彩斑斓？"):
    print(chunk.text, end="|", flush=True)

||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

In [10]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(full.text)

# The
# The sky
# The sky is
# The sky is typically
# The sky is typically blue
# ...

print(full.content_blocks)
# [{"type": "text", "text": "The sky is typically blue..."}]
































































The
The sky
The sky is
The sky is usually
The sky is usually blue
The sky is usually blue during
The sky is usually blue during a
The sky is usually blue during a clear
The sky is usually blue during a clear day
The sky is usually blue during a clear day.
The sky is usually blue during a clear day. At
The sky is usually blue during a clear day. At sunrise
The sky is usually blue during a clear day. At sunrise or
The sky is usually blue during a clear day. At sunrise or sunset
The sky is usually blue during a clear day. At sunrise or sunset it
The sky is usually blue during a clear day. At sunrise or sunset it can
The sky is usually blue during a clear day. At sunrise or sunset it can look
The sky is usually blue during a clear day. At sunrise or sunset it can look orange
The sky is usually blue during a clear day. At sunrise or sunset it can look orange,
The sky is usually blue during a clear day. At sunrise or sunset it ca

#### 4.3 batch

In [13]:
responses = model.batch([
    "为什么鹦鹉的羽毛色彩斑斓",
    "什么是量子计算"
])
for response in responses:
    print(response.content, end="\n")

鹦鹉羽毛色彩斑斓，并不是单一原因造成的，而是**独特色素、结构色、性选择、社交识别和生态适应**共同作用的结果。也不是所有鹦鹉都五彩缤纷，很多种类其实以绿色为主，用来在树冠中伪装。

先从“颜色怎么来”说：  
- **红、黄、橙**：主要来自鹦鹉特有的一类色素——**鹦鹉素（psittacofulvins）**，它们能自己合成，不像很多鸟类那样必须从食物中获取类胡萝卜素。  
- **蓝、绿、紫、紫外、虹彩**：多来自羽毛内部的**结构色**。蓝光由角蛋白和空气的纳米结构散射产生；绿色常常是“黄色鹦鹉素 + 蓝色结构色”叠加的结果。  
- **黑、灰、褐**：主要来自黑色素。

再从进化功能说：  
1. **求偶与性选择**：鲜艳羽毛常是健康和基因质量的信号。羽毛颜色饱满、结构完整，说明个体营养好、抗寄生虫能力强，更容易被配偶选中。  
2. **社交识别**：鹦鹉高度群居、寿命长、社会关系复杂。不同颜色和图案像“身份证”，帮助识别同种、个体、配偶和幼鸟，也能减少不同物种之间的杂交。  
3. **视觉交流**：鹦鹉有高度发达的色觉，许多种类还能看到紫外光。人类看到的颜色只是它们信号系统的一部分，它们眼中的羽毛可能更复杂、更醒目。  
4. **伪装与生态适应**：绿色羽毛让许多鹦鹉融入树叶，躲避猛禽。亮色在热带花果斑驳的环境中也可能打散轮廓，或在飞行、炫耀时传递信息。  
5. **捕食与繁殖的权衡**：太鲜艳容易被捕食者发现，所以自然选择和性选择会拉扯。群居、快速飞行、树冠生活等能在一定程度上降低鲜艳带来的风险。

所以，鹦鹉的斑斓色彩既是它们特殊“调色盘”的产物，也是进化中为了求偶、识别、交流和适应环境而形成的视觉语言。
量子计算是一种**利用量子力学规律来存储、处理和读取信息的计算范式**。它和经典计算机最根本的区别在于：经典计算机用“比特”（bit）表示 0 或 1；量子计算机用量子比特（qubit）表示信息，而量子比特可以处于 0 和 1 的**叠加态**。

一个量子比特可以写成：

\[
|\psi\rangle = \alpha |0\rangle + \beta |1\rangle
\]

其中 \(\alpha\)、\(\beta\) 是复数概率幅，测量时得到 0 的概率是 \(|\alpha|^2\)，得到 1 的概率是 \(|\bet